# sai regression
the same site regressions driven by the cesm2 change in light and diffuse fraction, g6 1.5k sai minus ssp2 4.5 for 2065 to 2084, with fit days and model years resampled (figure 6)

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import statsmodels.api as sm

data, out = Path("data"), Path("results")
out.mkdir(exist_ok=True)
drivers = ["PAR", "DF", "COSZ", "VPD", "TA", "SWC"]


def design(v):
    x = pd.DataFrame({k: np.asarray(v[k], float) for k in drivers})
    x["PAR"] = x.PAR / 1000
    x["PAR2"] = x.PAR ** 2
    x["PARxDF"] = x.PAR * x.DF
    return sm.add_constant(x[["PAR", "PAR2", "DF", "PARxDF", "COSZ", "VPD", "TA", "SWC"]], has_constant="add")

sites = {"US-Bi2": "maize", "US-Ne1": "maize", "US-Ne2": "maize", "US-Ne3": "maize",
         "US-xUN": "forest", "US-UMB": "forest", "US-MMS": "forest", "US-Ha1": "forest"}
# monthly means per run, site and year of the daily cesm2 values from 0b_cesm_extraction
c = pd.read_csv(data / "cesm_daily.csv", parse_dates=["date"]).dropna(subset=["FSDS"])
c = c.assign(DIF=c.SOLSD + c.SOLLD, month=c.date.dt.month, year=c.date.dt.year)
runs = {r: g.groupby(["site", "month", "year"])[["FSDS", "DIF"]].mean() for r, g in c.groupby("run")}
years = sorted(c.year.unique())

In [2]:
def deltas(ys_sai, ys_ssp):
    mean = lambda t, ys: pd.concat([t.xs(y, level="year", drop_level=False) for y in ys]).groupby(["site", "month"]).mean()
    a, b = mean(runs["SAI"], ys_sai), mean(runs["SSP"], ys_ssp)
    return pd.DataFrame({"fPAR": a.FSDS / b.FSDS, "dDF": a.DIF / a.FSDS - b.DIF / b.FSDS})


def effects(fit, d, dl, site):
    k = dl.loc[site].reindex(d.month.values)
    v0 = {x: d[x].values for x in drivers}
    f = lambda par, df: fit.predict(design({**v0, "PAR": par, "DF": df})).values
    p0, d0 = v0["PAR"], v0["DF"]
    p1, d1 = p0 * k.fPAR.values, np.clip(d0 + k.dDF.values, 0, 1)
    f00, f11, f10, f01 = f(p0, d0), f(p1, d1), f(p1, d0), f(p0, d1)
    light, diffuse = 0.5 * ((f10 - f00) + (f11 - f01)).mean(), 0.5 * ((f01 - f00) + (f11 - f10)).mean()
    g = d.GPP.mean()
    return {"light": 100 * light / g, "diffuse": 100 * diffuse / g, "radiation": 100 * (light + diffuse) / g}

In [3]:
dl = deltas(years, years)
rng = np.random.default_rng(2065)
draws, point = [], []
for site in sites:
    d = pd.read_csv(data / f"site_{site}.csv", parse_dates=["date"])
    d["month"] = d.date.dt.month
    d = d[d.month.isin(dl.loc[site].index)]
    x, y = design(d), d.GPP.values
    e = effects(sm.OLS(y, x).fit(), d, dl, site)
    point += [{"site": site, "component": k, "pct": v} for k, v in e.items()]
    days, idx = d.date.unique(), d.groupby("date").indices
    for _ in range(300):
        i = np.concatenate([idx[k] for k in rng.choice(days, len(days))])
        db = deltas(list(rng.choice(years, len(years))), list(rng.choice(years, len(years))))
        draws.append({"site": site, **effects(sm.OLS(y[i], x.iloc[i]).fit(), d, db, site)})

pd.DataFrame(draws).to_csv(out / "sai_draws.csv", index=False)
point = pd.DataFrame(point)
point.to_csv(out / "sai_point.csv", index=False)
point.pivot(index="site", columns="component", values="pct").round(1)

Out[3]: 
component  diffuse  light  radiation
site                                
US-Bi2         2.2   -0.6        1.7
US-Ha1         1.9    0.0        1.9
US-MMS         1.3   -0.9        0.4
US-Ne1         5.1   -4.1        1.0
US-Ne2         6.5   -5.2        1.4
US-Ne3         3.3   -3.5       -0.2
US-UMB         2.3   -0.9        1.4
US-xUN         4.8   -1.6        3.2


component,diffuse,light,radiation
site,,,
US-Bi2,2.2,-0.6,1.7
US-Ha1,1.9,0.0,1.9
US-MMS,1.3,-0.9,0.4
US-Ne1,5.1,-4.1,1.0
US-Ne2,6.5,-5.2,1.4
US-Ne3,3.3,-3.5,-0.2
US-UMB,2.3,-0.9,1.4
US-xUN,4.8,-1.6,3.2
